# Descarga del Dataset

In [6]:
from pathlib import Path

from huggingface_hub import snapshot_download

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

ruta = snapshot_download(
    repo_id="phamquiluan/RCAEval",
    repo_type="dataset",
    allow_patterns=["cases.parquet", "re2ss_*/metrics.parquet"],
    local_dir=project_root / "data" / "RCAEval",
)

print(ruta)

Fetching 91 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 91/91 [00:51<00:00,  1.76it/s]

/home/lambda/Obsidian/Segundo Cerebro/1.Proyectos/2026-2/Fundamentos de Inteligencia artificial/RandomForest-Implementation/data/RCAEval


## Load RE2-SS

In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

BASE = project_root / "data" / "RCAEval"
idx = pd.read_parquet(BASE / "cases.parquet")
re2 = idx.loc[idx["dataset"].eq("RE2-SS")].copy()
metrics_by_case = {}
for case_id in re2["case"]:
    metrics_path = BASE / case_id / "metrics.parquet"
    if metrics_path.is_file():
        metrics_by_case[case_id] = pd.read_parquet(metrics_path)


## Preview headers, row counts, and targets

In [6]:
print(f"Index rows (RE2-SS): {len(re2):,}")
display(re2.head())

print(f"Cases with local metrics: {len(metrics_by_case):,} / {len(re2):,}")
print(f"Total metric rows: {sum(len(frame) for frame in metrics_by_case.values()):,}")
if metrics_by_case:
    sample_case, sample_metrics = next(iter(metrics_by_case.items()))
    print(f"Metric preview: {sample_case} ({len(sample_metrics):,} rows)")
    display(sample_metrics.head())
else:
    print("No local RE2-SS metrics found. Run the download cell first.")

print("Target distribution: root_cause_service (one label per indexed case)")
display(re2["root_cause_service"].value_counts(dropna=False).rename("cases"))


Index rows (RE2-SS): 90


,case,dataset,suite,system,system_name,root_cause_service,fault,fault_description,repetition,inject_time,...,time_start,time_end,duration_minutes,normal_timesteps,faulty_timesteps,has_logs,n_logs,has_traces,n_traces,has_root_cause_file
465,re2ss_carts_cpu_1,RE2-SS,RE2,ss,Sock Shop,carts,cpu,CPU stress,1,1705596179,...,1705595459,1705596899,24.0,720,721,True,86592,False,0,False
466,re2ss_carts_cpu_2,RE2-SS,RE2,ss,Sock Shop,carts,cpu,CPU stress,2,1705676689,...,1705675969,1705677409,24.0,720,721,True,86451,False,0,False
467,re2ss_carts_cpu_3,RE2-SS,RE2,ss,Sock Shop,carts,cpu,CPU stress,3,1705678280,...,1705677560,1705679000,24.0,720,721,True,87020,False,0,False
468,re2ss_carts_delay_1,RE2-SS,RE2,ss,Sock Shop,carts,delay,network delay,1,1705662624,...,1705661904,1705663344,24.0,720,721,True,83040,False,0,False
469,re2ss_carts_delay_2,RE2-SS,RE2,ss,Sock Shop,carts,delay,network delay,2,1705725124,...,1705724404,1705725844,24.0,720,721,True,83145,False,0,False


Cases with local metrics: 90 / 90
Total metric rows: 129,690
Metric preview: re2ss_carts_cpu_1 (1,441 rows)


,time,carts_cpu,carts-db_cpu,catalogue_cpu,catalogue-db_cpu,front-end_cpu,orders_cpu,orders-db_cpu,payment_cpu,queue-master_cpu,...,payment_latency-50,shipping_latency-50,user_latency-50,carts_latency-90,catalogue_latency-90,front-end_latency-90,orders_latency-90,payment_latency-90,shipping_latency-90,user_latency-90
0,1705595459,1.163247,3.702944,0.170787,0.197159,4.500225,1.320988,0.369098,0.077473,0.422779,...,0.001966,0.003023,0.003369,0.027965,0.004662,0.069181,0.047346,0.004393,0.004641,0.006831
1,1705595460,1.139643,3.702944,0.170787,0.197159,4.500225,1.320988,0.369098,0.077473,0.422779,...,0.001966,0.003023,0.003345,0.027965,0.004662,0.069181,0.047346,0.004393,0.004641,0.006631
2,1705595461,1.116039,3.702944,0.170787,0.197159,4.543194,1.320988,0.369098,0.077473,0.413409,...,0.001966,0.003023,0.003345,0.027965,0.004662,0.069181,0.047346,0.004393,0.004641,0.006631
3,1705595462,1.576942,3.702944,0.170787,0.197159,4.543194,1.320988,0.369194,0.077473,0.413409,...,0.001966,0.003023,0.003345,0.027965,0.004659,0.069181,0.047346,0.004393,0.004641,0.006631
4,1705595463,1.661022,3.533458,0.170787,0.197159,4.543194,1.320988,0.369194,0.077473,0.413409,...,0.001966,0.003023,0.003345,0.027965,0.004659,0.066687,0.047346,0.004393,0.004641,0.006631


Target distribution: root_cause_service (one label per indexed case)


root_cause_service
carts        18
catalogue    18
orders       18
payment      18
user         18
Name: cases, dtype: int64

## Build clean feature and target tables

One row per incident and observed service. Split whole incidents by fault type before fitting column selection and imputation. These exploratory features summarize the full recording; use a diagnosis cutoff before evaluating an early diagnosis.

In [7]:
import numpy as np

if set(metrics_by_case) != set(re2["case"]):
    raise ValueError("Load all indexed RE2-SS cases before building the tables.")

allowed_metrics = {"cpu", "mem", "socket", "error", "latency-50", "latency-90"}
feature_rows = []
for incident_id, frame in metrics_by_case.items():
    service_features = {}
    for column in frame.columns:
        service, separator, metric = column.rpartition("_")
        if not separator or metric not in allowed_metrics:
            continue
        values = pd.to_numeric(frame[column], errors="raise").replace([np.inf, -np.inf], np.nan)
        features = service_features.setdefault(service, {})
        observed = values.dropna()
        if not observed.empty:
            features[f"{metric}_median"] = observed.median()
            features[f"{metric}_range"] = observed.max() - observed.min()
    for service, features in service_features.items():
        feature_rows.append({"incident_id": incident_id, "service": service, **features})

raw_features = pd.DataFrame(feature_rows).set_index(["incident_id", "service"], verify_integrity=True)
raw_features = raw_features.replace([np.inf, -np.inf], np.nan)

root_by_case = re2.set_index("case", verify_integrity=True)["root_cause_service"]
if root_by_case.isna().any():
    raise ValueError("Missing root-cause labels in the RE2-SS index.")
incident_ids = raw_features.index.get_level_values("incident_id")
services = raw_features.index.get_level_values("service")
y = pd.DataFrame({"is_root_cause": (services == incident_ids.map(root_by_case)).astype("int8")}, index=raw_features.index)
positives_by_case = y.groupby(level="incident_id")["is_root_cause"].sum()
if set(positives_by_case.index) != set(root_by_case.index) or not positives_by_case.eq(1).all():
    raise ValueError("Each case must contain its labeled root cause exactly once.")

split_rng = np.random.default_rng(42)
if re2["fault"].isna().any() or re2.groupby("fault").size().lt(2).any():
    raise ValueError("Each fault type must have at least two labeled incidents.")
test_cases = []
for _, fault_cases in re2.groupby("fault", sort=True):
    shuffled_cases = split_rng.permutation(fault_cases["case"].sort_values().to_numpy())
    n_test = max(1, round(0.20 * len(shuffled_cases)))
    test_cases.extend(shuffled_cases[:n_test])
test_mask = incident_ids.isin(test_cases)
train_raw = raw_features.loc[~test_mask]
test_raw = raw_features.loc[test_mask]
feature_columns = train_raw.columns[train_raw.nunique(dropna=True).gt(1)]
train_medians = train_raw[feature_columns].median()

X_train = train_raw[feature_columns].fillna(train_medians)
X_tests = test_raw[feature_columns].fillna(train_medians)
Y_train = y.loc[X_train.index]
Y_test = y.loc[X_tests.index]

assert X_train.shape[1] > 0 and X_train.nunique().gt(1).all()
assert X_train.columns.equals(X_tests.columns)
assert not set(X_train.index.get_level_values("incident_id")) & set(X_tests.index.get_level_values("incident_id"))
for name, features, targets in [("Train", X_train, Y_train), ("Test", X_tests, Y_test)]:
    assert features.index.equals(targets.index) and np.isfinite(features.to_numpy()).all()
    print(f"{name}: {len(features):,} rows, {features.shape[1]} features, {targets['is_root_cause'].sum():,} root causes")
split_labels = np.where(re2["case"].isin(test_cases), "Test", "Train")
display(pd.crosstab(re2["fault"], split_labels).reindex(columns=["Train", "Test"]))
display(X_train.head(), Y_train.head())


X: 1,352 rows, 12 features
Removed features: []
Imputed feature cells: 5,174
y: 1,352 rows, 90 positives


/tmp/ipykernel_1052396/4248589561.py:23: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  raw_features = pd.DataFrame(feature_rows).set_index(["incident_id", "service"], verify_integrity=True)
/tmp/ipykernel_1052396/4248589561.py:29: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  root_by_case = re2.set_index("case", verify_integrity=True)["root_cause_service"]


cpu_median  cpu_range   mem_median  \
incident_id       service                                            
re2ss_carts_cpu_1 carts           2.645536  49.237849  207855616.0   
                  carts-db        3.381748   2.742473   80842752.0   
                  catalogue       0.160062   0.106800    6369280.0   
                  catalogue-db    0.198928   0.122145  430735360.0   
                  front-end       4.264952   2.124096  102551552.0   

                                 mem_range  socket_median  socket_range  \
incident_id       service                                                 
re2ss_carts_cpu_1 carts          7213056.0           12.0           6.0   
                  carts-db       6172672.0            6.0           1.0   
                  catalogue       122880.0            6.0           0.0   
                  catalogue-db      8192.0            4.0           0.0   
                  front-end     13299712.0            8.0           3.0   

                                latency-50_median  latency-50_range  \
incident_id       service                                             
re2ss_carts_cpu_1 carts                  0.015581          0.028963   
                  carts-db               0.004456          0.003205   
                  catalogue              0.003022          0.000072   
                  catalogue-db           0.004456          0.003205   
                  front-end              0.029588          0.010175   

                                latency-90_median  latency-90_range  \
incident_id       service                                             
re2ss_carts_cpu_1 carts                  0.027733          0.065893   
                  carts-db               0.008935          0.011704   
                  catalogue              0.004640          0.000129   
                  catalogue-db           0.008935          0.011704   
                  front-end              0.065801          0.040209   

                                error_median  error_range  
incident_id       service                                  
re2ss_carts_cpu_1 carts                  0.0          0.0  
                  carts-db               0.0          0.0  
                  catalogue              0.0          0.0  
                  catalogue-db           0.0          0.0  
                  front-end              0.0          0.0

is_root_cause
incident_id       service                    
re2ss_carts_cpu_1 carts                     1
                  carts-db                  0
                  catalogue                 0
                  catalogue-db              0
                  front-end                 0

In [8]:
display(X.join(y).loc["re2ss_carts_cpu_1"])

,cpu_median,cpu_range,mem_median,mem_range,socket_median,socket_range,latency-50_median,latency-50_range,latency-90_median,latency-90_range,error_median,error_range,is_root_cause
service,,,,,,,,,,,,,
carts,2.645536,49.237849,207855616.0,7213056.0,12.0,6.0,0.015581,0.028963,0.027733,0.065893,0.0,0.0,1
carts-db,3.381748,2.742473,80842752.0,6172672.0,6.0,1.0,0.004456,0.003205,0.008935,0.011704,0.0,0.0,0
catalogue,0.160062,0.106800,6369280.0,122880.0,6.0,0.0,0.003022,0.000072,0.004640,0.000129,0.0,0.0,0
catalogue-db,0.198928,0.122145,430735360.0,8192.0,4.0,0.0,0.004456,0.003205,0.008935,0.011704,0.0,0.0,0
front-end,4.264952,2.124096,102551552.0,13299712.0,8.0,3.0,0.029588,0.010175,0.065801,0.040209,0.0,0.0,0
orders,1.176663,2.482078,327540736.0,876544.0,18.0,8.0,0.031875,0.041888,0.047056,0.052300,0.0,0.0,0
orders-db,0.360188,0.246544,40853504.0,2326528.0,9.0,4.0,0.004456,0.003205,0.008935,0.011704,0.0,0.0,0
payment,0.079242,0.042678,4489216.0,323584.0,5.0,0.0,0.001825,0.001170,0.004374,0.000253,0.0,0.0,0
queue-master,0.362695,3.620866,265396224.0,1941504.0,3.0,0.0,0.004456,0.003205,0.008935,0.011704,0.0,0.0,0


## Prepare one training sample per tree

Each tree gets a bootstrap sample of training rows and a fixed random subset of features. Test incidents are not used.

In [9]:
import numpy as np

def show_tree_samples(samples):
    display(pd.DataFrame({"tree": range(1, len(samples) + 1), "features": [list(features.columns) for features, _ in samples]}))


10 samples; 1,352 rows and 3 of 12 features per tree.


,tree,features
0,1,"[mem_median, socket_median, error_median]"
1,2,"[latency-90_median, cpu_median, error_range]"
2,3,"[socket_median, error_range, latency-90_median]"
3,4,"[mem_range, mem_median, error_range]"
4,5,"[cpu_range, latency-90_median, socket_median]"
5,6,"[error_median, latency-90_range, mem_median]"
6,7,"[socket_range, latency-50_median, socket_median]"
7,8,"[error_range, error_median, latency-50_median]"
8,9,"[socket_median, latency-50_range, error_median]"
9,10,"[latency-50_median, cpu_median, mem_range]"


---

## Random Forest training

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass


@dataclass(frozen=True)
class _TreeNode:
    positive_fraction: float
    feature: str | None = None
    threshold: float | None = None
    left: '_TreeNode | None' = None
    right: '_TreeNode | None' = None


class RandomForest:
    """Train binary CART trees for scoring candidate root causes."""

    def _prepare_tree_samples(
        self, X_train: pd.DataFrame, Y_train: pd.DataFrame, *, n_trees: int = 10, seed: int = 42
    ) -> list[tuple[pd.DataFrame, pd.DataFrame]]:
        """Draw bootstrap rows and a fixed feature subset for each tree."""
        if n_trees < 1 or X_train.empty or Y_train.empty or not X_train.index.equals(Y_train.index):
            raise ValueError("Provide at least one tree and nonempty, aligned training tables.")

        rng = np.random.default_rng(seed)
        n_rows, k = X_train.shape
        n_features_per_tree = int(np.sqrt(k))
        samples = []
        for _ in range(n_trees):
            rows = rng.choice(n_rows, size=n_rows, replace=True)
            features = rng.choice(k, size=n_features_per_tree, replace=False)
            samples.append((X_train.iloc[rows, features].copy(), Y_train.iloc[rows].copy()))
        return samples

    def _build_tree(self, X_tree: pd.DataFrame, y_tree: pd.DataFrame) -> _TreeNode:
        """Grow one binary CART tree using only its assigned features."""
        values = X_tree.to_numpy(dtype=float)
        if y_tree.shape[1] != 1 or not y_tree.isin([0, 1]).to_numpy().all():
            raise ValueError("Provide exactly one binary target column.")
        if not np.isfinite(values).all():
            raise ValueError("Tree features must be finite; impute missing values before fitting.")
        labels = y_tree.iloc[:, 0].to_numpy(dtype=np.int64)
        max_depth, min_leaf = 8, 2

        def grow(rows: np.ndarray, depth: int) -> _TreeNode:
            size = len(rows)
            positives = labels[rows].sum()
            fraction = float(positives / size)
            leaf = _TreeNode(fraction)
            if depth >= max_depth or size < 2 * min_leaf or positives in (0, size):
                return leaf

            best_impurity = 2 * fraction * (1 - fraction)
            best_split = None
            for feature in range(values.shape[1]):
                ordered_rows = rows[np.argsort(values[rows, feature], kind="stable")]
                ordered_values = values[ordered_rows, feature]
                cuts = np.flatnonzero(ordered_values[:-1] < ordered_values[1:]) + 1
                cuts = cuts[(cuts >= min_leaf) & (size - cuts >= min_leaf)]
                if cuts.size == 0:
                    continue

                left_size, right_size = cuts, size - cuts
                left_pos = np.cumsum(labels[ordered_rows])[cuts - 1]
                right_pos = positives - left_pos
                left_rate, right_rate = left_pos / left_size, right_pos / right_size
                impurities = 2 * (
                    left_size * left_rate * (1 - left_rate)
                    + right_size * right_rate * (1 - right_rate)
                ) / size
                candidate = int(np.argmin(impurities))
                if impurities[candidate] < best_impurity - 1e-12:
                    best_impurity = impurities[candidate]
                    best_split = feature, ordered_rows, int(cuts[candidate])

            if best_split is None:
                return leaf
            feature, ordered_rows, cut = best_split
            lower, upper = values[ordered_rows[[cut - 1, cut]], feature]
            threshold = lower / 2 + upper / 2
            # Keep the split valid if the midpoint rounds to the upper value.
            if not lower <= threshold < upper:
                threshold = lower
            return _TreeNode(
                fraction, X_tree.columns[feature], float(threshold),
                grow(ordered_rows[:cut], depth + 1),
                grow(ordered_rows[cut:], depth + 1),
            )

        return grow(np.arange(len(labels)), depth=0)

    def fit(self, X_train: pd.DataFrame, Y_train: pd.DataFrame) -> "RandomForest":
        """Prepare aligned bootstrap samples and build their trees concurrently."""
        samples = self._prepare_tree_samples(X_train, Y_train)
        with ThreadPoolExecutor() as executor:
            trees = list(executor.map(self._build_tree, *zip(*samples)))
        self._tree_samples, self._trees = samples, trees
        return self

    def predict(self, x: pd.Series) -> float:
        """Return the root-cause score for one service feature row."""
        raise NotImplementedError

    def export(self, path: str | Path) -> None:
        """Save the fitted model to the given path."""
        raise NotImplementedError

    @classmethod
    def from_export(cls, path: str | Path) -> "RandomForest":
        """Construct a forest from a model saved with export."""
        raise NotImplementedError


In [ ]:
# Fit the forest and inspect the feature subset assigned to each tree.
forest = RandomForest().fit(X_train, Y_train)
show_tree_samples(forest._tree_samples)
print(f"Built {len(forest._trees)} trees.")


## Detailed exploration

In [11]:

import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)


def titulo(txt):
    print("\n" + "=" * 75)
    print(txt)
    print("=" * 75)


def resumen(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "nulos_%": (
            df.isna().mean() * 100
        ).round(2),
        "unicos": df.nunique(dropna=True),
    })


def muestra(df, nombre, n=5):
    titulo(nombre)
    print("Dimensiones:", df.shape)
    print("Memoria (MiB):", round(
        df.memory_usage(deep=True).sum()
        / 1024**2, 2
    ))
    display(resumen(df))
    display(df.head(n))


titulo("1. INFORMACIÓN GENERAL")
print("Casos del subconjunto:", len(re2))
print("Columnas del índice:", len(re2.columns))
display(re2.head())
display(resumen(re2))

# 2. Etiquetas y distribución
titulo("2. TIPOS DE FALLAS")
display(
    re2["fault"].value_counts()
    .rename_axis("falla")
    .reset_index(name="casos")
)

titulo("3. SERVICIOS CAUSANTES")
display(
    re2["root_cause_service"]
    .value_counts()
    .rename_axis("servicio")
    .reset_index(name="casos")
)

titulo("4. FALLAS POR SERVICIO")
display(pd.crosstab(
    re2["root_cause_service"],
    re2["fault"]
))

# 3. Cobertura general
titulo("5. COBERTURA DEL DATASET")
cols = [
    "n_metrics", "n_timesteps",
    "duration_minutes",
    "normal_timesteps",
    "faulty_timesteps",
    "n_logs", "n_traces"
]
cols = [c for c in cols if c in re2.columns]

display(re2[cols].describe().T)

# 4. Archivos disponibles
titulo("6. ARCHIVOS DESCARGADOS")

filas = []
for _, fila in re2.iterrows():
    carp = BASE / fila["case"]
    reg = {"caso": fila["case"]}
    for nombre in [
        "metrics.parquet",
        "logs.parquet",
        "traces.parquet",
        "inject_time.txt"
    ]:
        arch = carp / nombre
        reg[nombre] = arch.exists()
        reg[nombre + "_MiB"] = (
            round(arch.stat().st_size / 1024**2, 2)
            if arch.exists() else np.nan
        )
    filas.append(reg)

archivos = pd.DataFrame(filas)
display(archivos.head(10))

print("Disponibilidad por archivo:")
display(
    archivos[
        [c for c in archivos.columns
         if c.endswith((".parquet", ".txt"))]
    ].sum()
)

# 5. Seleccionar un caso disponible
validos = [
    c for c in re2["case"]
    if (BASE / c / "metrics.parquet").exists()
]

if not validos:
    raise FileNotFoundError(
        "No RE2-SS metrics found. Run the download cell first."
    )

caso = validos[0]
carp = BASE / caso
meta = re2.loc[re2["case"] == caso].iloc[0]

titulo("7. CASO SELECCIONADO")
print("Caso:", caso)
print("Servicio raíz:", meta["root_cause_service"])
print("Tipo de falla:", meta["fault"])
print("Repetición:", meta["repetition"])
print("Inyección Unix:", meta["inject_time"])
print("Inyección UTC:", pd.to_datetime(
    meta["inject_time"], unit="s", utc=True
))

# 6. Métricas
met = metrics_by_case[caso]
muestra(met, "8. ESTRUCTURA DE MÉTRICAS")

titulo("9. ESTADÍSTICAS DE MÉTRICAS")
num = met.select_dtypes(include=np.number)
num = num.drop(columns=["time"], errors="ignore")

display(num.describe().T)

titulo("10. MÉTRICAS CONSTANTES")
const = num.columns[
    num.nunique(dropna=True) <= 1
].tolist()
print("Cantidad:", len(const))
print(const)

titulo("11. MÉTRICAS CON MÁS NULOS")
display(
    resumen(num)
    .sort_values("nulos_%", ascending=False)
    .head(20)
)

# 7. Temporalidad
titulo("12. ANÁLISIS TEMPORAL")

if "time" in met.columns:
    t = pd.to_numeric(
        met["time"], errors="coerce"
    ).dropna().sort_values().drop_duplicates()

    print("Inicio UTC:", pd.to_datetime(
        t.min(), unit="s", utc=True
    ))
    print("Fin UTC:", pd.to_datetime(
        t.max(), unit="s", utc=True
    ))

    dt = t.diff().dropna()
    print("Intervalos (segundos):")
    display(dt.describe())

    iny = int(meta["inject_time"])
    normal = met[met["time"] < iny]
    falla = met[met["time"] >= iny]

    print("Filas normales:", len(normal))
    print("Filas posteriores:", len(falla))

# 8. Comparación antes/después
titulo("13. CAMBIOS EN MÉTRICAS")

if len(normal) and len(falla):
    a = normal[num.columns].mean()
    b = falla[num.columns].mean()

    comp = pd.DataFrame({
        "media_normal": a,
        "media_posterior": b,
        "diferencia": b - a,
    })
    comp["cambio_abs"] = (
        comp["diferencia"].abs()
    )

    display(
        comp.sort_values(
            "cambio_abs", ascending=False
        ).head(20)
    )

# 9. Logs y trazas
for nombre in ["logs", "traces"]:
    arch = carp / f"{nombre}.parquet"

    if arch.exists():
        df = pd.read_parquet(arch)
        muestra(
            df,
            f"14. ESTRUCTURA DE {nombre.upper()}"
        )

        if nombre == "traces":
            for col in ["serviceName", "statusCode"]:
                if col in df.columns:
                    print(f"Distribución de {col}:")
                    display(
                        df[col].value_counts().head(20)
                    )
    else:
        print(f"No existe {nombre}.parquet")

# 10. Compatibilidad entre casos
titulo("15. CONSISTENCIA DE CARACTERÍSTICAS")

esquemas = {}
for c in validos:
    columnas = metrics_by_case[c].columns
    esquemas[c] = set(columnas) - {"time"}

union = set().union(*esquemas.values())
inter = set.intersection(*esquemas.values())

print("Casos inspeccionados:", len(esquemas))
print("Métricas distintas:", len(union))
print("Métricas comunes:", len(inter))
print("Métricas no comunes:", len(union - inter))
print("Ejemplos de métricas comunes:")
print(sorted(inter)[:30])

titulo("EXPLORACIÓN FINALIZADA")



1. INFORMACIÓN GENERAL
Casos del subconjunto: 90
Columnas del índice: 22


,case,dataset,suite,system,system_name,root_cause_service,fault,fault_description,repetition,inject_time,n_metrics,n_timesteps,time_start,time_end,duration_minutes,normal_timesteps,faulty_timesteps,has_logs,n_logs,has_traces,n_traces,has_root_cause_file
465,re2ss_carts_cpu_1,RE2-SS,RE2,ss,Sock Shop,carts,cpu,CPU stress,1,1705596179,75,1441,1705595459,1705596899,24.0,720,721,True,86592,False,0,False
466,re2ss_carts_cpu_2,RE2-SS,RE2,ss,Sock Shop,carts,cpu,CPU stress,2,1705676689,78,1441,1705675969,1705677409,24.0,720,721,True,86451,False,0,False
467,re2ss_carts_cpu_3,RE2-SS,RE2,ss,Sock Shop,carts,cpu,CPU stress,3,1705678280,78,1441,1705677560,1705679000,24.0,720,721,True,87020,False,0,False
468,re2ss_carts_delay_1,RE2-SS,RE2,ss,Sock Shop,carts,delay,network delay,1,1705662624,79,1441,1705661904,1705663344,24.0,720,721,True,83040,False,0,False
469,re2ss_carts_delay_2,RE2-SS,RE2,ss,Sock Shop,carts,delay,network delay,2,1705725124,75,1441,1705724404,1705725844,24.0,720,721,True,83145,False,0,False


,tipo,nulos,nulos_%,unicos
case,str,0,0.0,90
dataset,str,0,0.0,1
suite,str,0,0.0,1
system,str,0,0.0,1
system_name,str,0,0.0,1
root_cause_service,str,0,0.0,5
fault,str,0,0.0,6
fault_description,str,0,0.0,6
repetition,int64,0,0.0,3
inject_time,int64,0,0.0,90



2. TIPOS DE FALLAS


,falla,casos
0,cpu,15
1,delay,15
2,disk,15
3,loss,15
4,mem,15
5,socket,15



3. SERVICIOS CAUSANTES


... (17 outputs left out)

count    1440.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: time, dtype: float64

Filas normales: 720
Filas posteriores: 721

13. CAMBIOS EN MÉTRICAS


,media_normal,media_posterior,diferencia,cambio_abs
carts_mem,2.072481e+08,2.130095e+08,5.761381e+06,5.761381e+06
carts-db_mem,7.943071e+07,8.240345e+07,2.972740e+06,2.972740e+06
user-db_mem,4.733726e+07,4.981075e+07,2.473490e+06,2.473490e+06
orders-db_mem,4.026441e+07,4.135652e+07,1.092108e+06,1.092108e+06
queue-master_mem,2.652533e+08,2.655816e+08,3.282702e+05,3.282702e+05
orders_mem,3.273551e+08,3.276304e+08,2.753428e+05,2.753428e+05
carts-db_diskio,1.933343e+06,2.185288e+06,2.519455e+05,2.519455e+05
front-end_mem,1.032873e+08,1.031104e+08,-1.769781e+05,1.769781e+05
rabbitmq_mem,1.002310e+08,1.003225e+08,9.152400e+04,9.152400e+04
shipping_mem,3.059457e+08,3.058620e+08,-8.372528e+04,8.372528e+04


No existe logs.parquet
No existe traces.parquet

15. CONSISTENCIA DE CARACTERÍSTICAS
Casos inspeccionados: 90
Métricas distintas: 86
Métricas comunes: 73
Métricas no comunes: 13
Ejemplos de métricas comunes:
['carts-db_cpu', 'carts-db_diskio', 'carts-db_mem', 'carts-db_socket', 'carts_cpu', 'carts_latency-50', 'carts_latency-90', 'carts_mem', 'carts_socket', 'carts_workload', 'catalogue-db_cpu', 'catalogue-db_diskio', 'catalogue-db_mem', 'catalogue-db_socket', 'catalogue_cpu', 'catalogue_latency-50', 'catalogue_latency-90', 'catalogue_mem', 'catalogue_socket', 'catalogue_workload', 'front-end_cpu', 'front-end_latency-50', 'front-end_latency-90', 'front-end_mem', 'front-end_socket', 'front-end_workload', 'orders-db_cpu', 'orders-db_diskio', 'orders-db_mem', 'orders-db_socket']

EXPLORACIÓN FINALIZADA
